## **Pipeline de Geração dos Embeddings utilizados pelo RAG**

**Dataset:** [AiresPucrs/tmdb-5000-movies](https://huggingface.co/datasets/AiresPucrs/tmdb-5000-movies) — ~4.800 filmes com elenco, diretores, gêneros e resumo

**Etapas:**
1. Instalação das dependências
2. ETL — carga e limpeza do dataset
3. Criação dos chunks de texto por filme
4. Geração dos embeddings (all-MiniLM-L6-v2)
5. Serialização em numpy + pickle
6. Verificação via busca por similaridade coseno

### **0 - Instalação das dependências**

In [ ]:
%pip install datasets sentence-transformers pandas numpy tqdm scikit-learn

### **1 - Imports**

In [22]:
import ast
import json
import pickle
import os

import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print('Imports OK')

Imports OK


### **2 - ETL: Carregar e Limpar o Dataset**

In [18]:
raw = load_dataset('AiresPucrs/tmdb-5000-movies', split='train')
df = raw.to_pandas()

print(f'Total de filmes carregados: {len(df)}')
print(f'Colunas disponíveis: {df.columns.tolist()}')
df.head(2)

Total de filmes carregados: 4803
Colunas disponíveis: ['id', 'budget', 'genres', 'homepage', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'cast', 'crew']


,id,budget,genres,homepage,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,5,4000000,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 35, ""name...",None,"[{""id"": 612, ""name"": ""hotel""}, {""id"": 613, ""na...",en,Four Rooms,It's Ted the Bellhop's first night on the job....,22.876230,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...",...,4300000,98.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Twelve outrageous guests. Four scandalous requ...,Four Rooms,6.5,530,"[{""cast_id"": 42, ""character"": ""Ted the Bellhop...","[{""credit_id"": ""52fe420dc3a36847f800012d"", ""de..."
1,11,11000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",http://www.starwars.com/films/star-wars-episod...,"[{""id"": 803, ""name"": ""android""}, {""id"": 4270, ...",en,Star Wars,Princess Leia is captured and held hostage by ...,126.393695,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...",...,775398007,121.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"A long time ago in a galaxy far, far away...",Star Wars,8.1,6624,"[{""cast_id"": 3, ""character"": ""Luke Skywalker"",...","[{""credit_id"": ""52fe420dc3a36847f8000437"", ""de..."


In [ ]:
def _parse_field(value):
    """Converte campo para lista de dicts, independente do formato de origem."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    # HuggingFace pode entregar o campo já como lista nativa
    if isinstance(value, list):
        return value
    raw = str(value)
    # Tenta JSON com aspas duplas primeiro
    try:
        return json.loads(raw)
    except Exception:
        pass
    # Fallback: formato Python repr com aspas simples (ast não quebra com apóstrofes)
    try:
        return ast.literal_eval(raw)
    except Exception:
        return []


def parse_json_names(value, key='name', top_n=None):
    """Extrai lista de nomes de um campo JSON/repr de lista de dicts."""
    items = _parse_field(value)
    names = [item[key] for item in items if isinstance(item, dict) and key in item]
    return names[:top_n] if top_n else names


def parse_directors(value):
    """Extrai nomes dos diretores da coluna crew (job == 'Director')."""
    items = _parse_field(value)
    return [m['name'] for m in items if isinstance(m, dict) and m.get('job') == 'Director']


# 1. Dropar linhas sem título ou resumo
df = df.dropna(subset=['title', 'overview'])
df = df[df['title'].str.strip() != '']
df = df[df['overview'].str.strip() != '']

# 2. Filtrar filmes sem avaliações confiáveis
df = df[df['vote_count'] >= 10]

# 3. Extrair ano de release_date
df['year'] = pd.to_numeric(df['release_date'].str[:4], errors='coerce').fillna(0).astype(int)

# 4. Parsear genres → lista de nomes
df['genres_list'] = df['genres'].apply(lambda x: parse_json_names(x, key='name'))

# 5. Parsear cast → top-5 atores principais
df['cast_list'] = df['cast'].apply(lambda x: parse_json_names(x, key='name', top_n=5))

# 6. Parsear crew → diretores
df['director_list'] = df['crew'].apply(parse_directors)

# 7. Garantir vote_average numérico
df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce').fillna(0.0)

df = df.reset_index(drop=True)
print(f'Filmes após limpeza: {len(df)}')
print(f'Exemplo Titulo:      {df["title"].iloc[0]}')
print(f'Exemplo de elenco:   {df["cast_list"].iloc[0]}')
print(f'Exemplo de diretor:  {df["director_list"].iloc[0]}')
print(f'Exemplo de gêneros:  {df["genres_list"].iloc[0]}')

Filmes após limpeza: 4391
Exemplo Titulo:   Four Rooms
Exemplo de elenco:   ['Tim Roth', 'Antonio Banderas', 'Jennifer Beals', 'Madonna', 'Marisa Tomei']
Exemplo de diretor:  ['Allison Anders', 'Alexandre Rockwell', 'Robert Rodriguez', 'Quentin Tarantino']
Exemplo de gêneros:  ['Crime', 'Comedy']


### **3 - Criação dos Chunks**

Cada filme vira um único chunk de texto. Formato:
```
Title: {title}
Year: {year}
Genres: {genre1}, {genre2}
Director: {director}
Rating: {vote_average}/10
Cast: {actor1}, {actor2}, ...
Overview: {overview}
```

O dicionário `metadata` armazena os campos como listas para filtros dos agentes (gênero, ator, diretor, ano).

In [27]:
chunks = []
metadata = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Criando chunks'):
    genres_str   = ', '.join(row['genres_list'])   if row['genres_list']   else 'Unknown'
    cast_str     = ', '.join(row['cast_list'])     if row['cast_list']     else 'Unknown'
    director_str = ', '.join(row['director_list']) if row['director_list'] else 'Unknown'
    year_str     = str(row['year'])                if row['year'] > 0      else 'Unknown'

    chunk = (
        f"Title: {row['title']}\n"
        f"Year: {year_str}\n"
        f"Genres: {genres_str}\n"
        f"Director: {director_str}\n"
        f"Rating: {row['vote_average']:.1f}/10\n"
        f"Cast: {cast_str}\n"
        f"Overview: {row['overview'].strip()}"
    )
    chunks.append(chunk)

    metadata.append({
        'id':                int(idx),
        'title':             str(row['title']),
        'year':              int(row['year']),
        'genres':            row['genres_list'],
        'director':          row['director_list'],
        'cast':              row['cast_list'],
        'vote_average':      float(row['vote_average']),
        'vote_count':        int(row['vote_count']),
        'original_language': str(row.get('original_language', '')),
        'overview':          str(row['overview']).strip(),
    })

print(f'Total de chunks criados: {len(chunks)}')
print(f'\nExemplo de chunk:\n{chunks[1]}')

Criando chunks: 100%|██████████| 4391/4391 [00:00<00:00, 9484.77it/s] 

Total de chunks criados: 4391

Exemplo de chunk:
Title: Star Wars
Year: 1977
Genres: Adventure, Action, Science Fiction
Director: George Lucas
Rating: 8.1/10
Cast: Mark Hamill, Harrison Ford, Carrie Fisher, Peter Cushing, Alec Guinness
Overview: Princess Leia is captured and held hostage by the evil Imperial forces in their effort to take over the galactic Empire. Venturesome Luke Skywalker and dashing captain Han Solo team together with the loveable robot duo R2-D2 and C-3PO to rescue the beautiful princess and restore peace and justice in the Empire.


### **4 - Geração dos Embeddings**

In [28]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print('Gerando embeddings (pode levar alguns minutos)...')
embeddings = model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f'Shape dos embeddings: {embeddings.shape}')  # esperado: (N, 384)
print(f'Dtype: {embeddings.dtype}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10266.48it/s]


Gerando embeddings (pode levar alguns minutos)...


Batches: 100%|██████████| 69/69 [00:49<00:00,  1.41it/s]

Shape dos embeddings: (4391, 384)
Dtype: float32


### **5 - Salvar Embeddings em RAG/data/**

Conteúdo salvo no formato numpy .npy e pickle .pkl

In [29]:
os.makedirs('data', exist_ok=True)

np.save('data/embeddings.npy', embeddings.astype(np.float32))

with open('data/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

with open('data/chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

print('Arquivos salvos em RAG/data/:')
for fname in os.listdir('data'):
    size_mb = os.path.getsize(f'data/{fname}') / 1024 / 1024
    print(f'  {fname}  ({size_mb:.2f} MB)')

Arquivos salvos em RAG/data/:
  chunks.pkl  (2.10 MB)
  embeddings.npy  (6.43 MB)
  metadata.pkl  (2.10 MB)


### **6 - Verificação: Busca por Similaridade Coseno**

Testa o pipeline completo. Resultados semanticamente coerentes confirmam que os embeddings estão corretos.

In [31]:
from sklearn.metrics.pairwise import cosine_similarity

emb_loaded = np.load('data/embeddings.npy')
with open('data/metadata.pkl', 'rb') as f:
    meta_loaded = pickle.load(f)
with open('data/chunks.pkl', 'rb') as f:
    chunks_loaded = pickle.load(f)

def search(query: str, top_n: int = 5):
    q_emb = model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, emb_loaded)[0]
    top_idx = np.argsort(scores)[::-1][:top_n]
    print(f'Query: "{query}"')
    print('=' * 60)
    for rank, i in enumerate(top_idx, 1):
        print(f'\n#{rank} — Score: {scores[i]:.4f}')
        print(chunks_loaded[i])
        print('-' * 60)

search('action movie with space adventure')
search('romantic comedy in Paris')
search('Christopher Nolan psychological thriller')

Query: "action movie with space adventure"

#1 — Score: 0.5470
Title: Gravity
Year: 2013
Genres: Science Fiction, Thriller, Drama
Director: Alfonso Cuarón
Rating: 7.3/10
Cast: Sandra Bullock, George Clooney, Ed Harris, Orto Ignatiussen, Phaldut Sharma
Overview: Dr. Ryan Stone, a brilliant medical engineer on her first Shuttle mission, with veteran astronaut Matt Kowalsky in command of his last flight before retiring. But on a seemingly routine spacewalk, disaster strikes. The Shuttle is destroyed, leaving Stone and Kowalsky completely alone-tethered to nothing but each other and spiraling out into the blackness of space. The deafening silence tells them they have lost any link to Earth and any chance for rescue. As fear turns to panic, every gulp of air eats away at what little oxygen is left. But the only way home may be to go further out into the terrifying expanse of space.
------------------------------------------------------------

#2 — Score: 0.5266
Title: U.F.O.
Year: 2012
Genr